# FIFA Player Data — In-Depth Analysis with Linear Regression
### What Drives Player Ratings and Market Values?

## 1. Project Overview
This notebook performs an in-depth exploratory and statistical analysis of FIFA player data, investigating the relationships between player attributes (pace, shooting, dribbling, etc.), overall rating, market value, and wage. We finish by building a linear regression model to predict overall rating from technical attributes.

## 2. Learning Objectives
- Explore multi-dimensional player attribute data
- Identify which attributes most strongly predict overall rating
- Understand position-specific attribute profiles
- Build and evaluate a linear regression model
- Visualise feature importance via correlation and regression coefficients

## 3. Business / Research Problem
**Questions:**
1. Which technical attributes best explain a player's overall FIFA rating?
2. Do attacking and defensive players have different attribute profiles?
3. Is player value better explained by rating or by age?
4. What is the relationship between overall rating and wage?

## 4. Why This Analysis Matters
Football clubs use performance analytics to identify undervalued players (Moneyball-style). Understanding what the FIFA rating algorithm weights most heavily helps scouts and analysts identify value gaps — players with high attribute scores but undervalued overall ratings.

## 5. Dataset Overview
Key columns include overall, potential, pace, shooting, passing, dribbling, defending, physic ratings, plus value_eur, wage_eur, age, and position.

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `stefanoleone992/fifa-22-complete-player-dataset`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'stefanoleone992/fifa-22-complete-player-dataset'
ATTR_COLS = ['pace','shooting','passing','dribbling','defending','physic']
TARGET = 'overall'

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset
License(s): CC0-1.0


Files: ['female_players_16.csv', 'female_players_17.csv', 'female_players_18.csv', 'female_players_19.csv', 'female_players_20.csv', 'female_players_21.csv', 'female_players_22.csv', 'players_15.csv', 'players_16.csv', 'players_17.csv', 'players_18.csv', 'players_19.csv', 'players_20.csv', 'players_21.csv', 'players_22.csv']


In [5]:
# Load male players file
male_file = [f for f in csv_files if 'male' in f.name.lower() and 'female' not in f.name.lower()]
if not male_file: male_file = csv_files
df = pd.read_csv(male_file[0], low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

Shape: (248, 110)


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,226324,https://sofifa.com/player/226324/carli-lloyd/1...,C. Lloyd,Carli Anne Hollins,"CM, CAM, LM, ST",91,91,NaN,NaN,32,...,82+3,82+3,82+3,83+3,19+3,https://cdn.sofifa.com/players/226/324/16_120.png,NaN,NaN,https://cdn.sofifa.com/teams/113009/60.png,https://cdn.sofifa.com/flags/us.png
1,226328,https://sofifa.com/player/226328/megan-rapinoe...,M. Rapinoe,Megan Anna Rapinoe,"LM, CM",90,90,NaN,NaN,29,...,56+3,56+3,56+3,66+3,19+3,https://cdn.sofifa.com/players/226/328/16_120.png,NaN,NaN,https://cdn.sofifa.com/teams/113009/60.png,https://cdn.sofifa.com/flags/us.png
2,226334,https://sofifa.com/player/226334/abby-wambach/...,A. Wambach,Abby Wambach,ST,90,90,NaN,NaN,35,...,55+3,55+3,55+3,55+3,20+3,https://cdn.sofifa.com/players/226/334/16_120.png,NaN,NaN,https://cdn.sofifa.com/teams/113009/60.png,https://cdn.sofifa.com/flags/us.png


## 11. Data Validation Checks

In [6]:
print('Columns:', df.columns.tolist()[:20])
available_attrs = [c for c in ATTR_COLS if c in df.columns]
print('Available attribute columns:', available_attrs)
print('\nMissing values (key cols):')
print(df[available_attrs + ['overall','value_eur','wage_eur']].isnull().sum())

Columns: ['sofifa_id', 'player_url', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur', 'age', 'dob', 'height_cm', 'weight_kg', 'club_team_id', 'club_name', 'league_name', 'league_level', 'club_position', 'club_jersey_number', 'club_loaned_from']
Available attribute columns: ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']

Missing values (key cols):
pace          33
shooting      33
passing       33
dribbling     33
defending     33
physic        33
overall        0
value_eur    248
wage_eur     248
dtype: int64


## 12. Data Cleaning

In [7]:
df = df.dropna(subset=['overall'])
for col in available_attrs:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=available_attrs)
print(f'Clean records: {len(df)}')

Clean records: 215


## 13. Exploratory Data Analysis

In [8]:
print('Overall rating stats:')
print(df['overall'].describe().round(1))
print('\nTop 10 players:')
name_col = 'short_name' if 'short_name' in df.columns else ('name' if 'name' in df.columns else df.columns[0])
print(df.nlargest(10,'overall')[[name_col,'overall','age']].to_string(index=False) if 'age' in df.columns else df.nlargest(10,'overall')[[name_col,'overall']].to_string(index=False))

Overall rating stats:
count    215.0
mean      76.0
std        5.4
min       61.0
25%       73.0
50%       76.0
75%       79.0
max       91.0
Name: overall, dtype: float64

Top 10 players:
   short_name  overall  age
     C. Lloyd       91   32
   M. Rapinoe       90   29
   A. Wambach       90   35
     L. Nécib       90   28
    N. Keßler       89   27
  C. Sinclair       88   32
        Marta       88   29
   L. Schelin       87   31
B. Sauerbrunn       86   30
  D. Marozsán       86   23


## 14. Univariate Analysis

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(16,8))
for ax, col in zip(axes.flat, available_attrs[:6]):
    ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col.title())
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1)
plt.suptitle('Distribution of Player Attribute Ratings', fontsize=14)
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate AnalysisThis section compares how player attributes move together and how rating, age, value, wage, and position interact.

In [10]:
corrs = df[available_attrs + ['overall']].corr()['overall'].drop('overall').sort_values(ascending=False)
print('Correlations with Overall:')
print(corrs.round(3))
fig, ax = plt.subplots(figsize=(9,4))
corrs.plot(kind='bar', ax=ax, color=['green' if v>0 else 'red' for v in corrs])
ax.set_title('Attribute Pearson Correlation with Overall Rating')
ax.set_ylabel('r')
plt.tight_layout(); plt.show()

Correlations with Overall:
passing      0.468
dribbling    0.435
shooting     0.337
physic       0.290
pace         0.210
defending    0.099
Name: overall, dtype: float64


## 16. Feature-Specific Insights — Position Profiles

In [11]:
if 'player_positions' in df.columns or 'best_overall_rating' in df.columns:
    pos_col = 'player_positions' if 'player_positions' in df.columns else None
else:
    pos_col = None
if pos_col:
    df['primary_pos'] = df[pos_col].str.split(',').str[0].str.strip()
    pos_groups = df.groupby('primary_pos')[available_attrs].mean()
    # Top 5 most common positions
    top_pos = df['primary_pos'].value_counts().head(5).index
    pos_groups = pos_groups.loc[pos_groups.index.isin(top_pos)]
    fig, ax = plt.subplots(figsize=(12,5))
    pos_groups.T.plot(ax=ax)
    ax.set_title('Average Attribute Ratings by Position')
    ax.set_ylabel('Rating')
    ax.legend(bbox_to_anchor=(1,1))
    plt.tight_layout(); plt.show()

## 17. Statistical Checks

In [12]:
r, p = stats.pearsonr(df['dribbling'], df['overall'])
print(f'Dribbling vs Overall: r={r:.3f}, p={p:.4e}')
r2, p2 = stats.pearsonr(df['defending'], df['overall'])
print(f'Defending vs Overall: r={r2:.3f}, p={p2:.4e}')

Dribbling vs Overall: r=0.435, p=2.5402e-11
Defending vs Overall: r=0.099, p=1.4968e-01


## 18. Visual Analysis### Linear Regression ModelPredict overall rating from the six main attribute categories.

In [13]:
X = df[available_attrs].fillna(df[available_attrs].mean())
y = df['overall']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
lr = LinearRegression()
lr.fit(X_train_s, y_train)
preds = lr.predict(X_test_s)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f'Linear Regression — R²: {r2:.3f}, RMSE: {rmse:.2f}')
coef_df = pd.Series(lr.coef_, index=available_attrs).sort_values(ascending=False)
print('\nRegression coefficients (standardised):')
print(coef_df.round(3))

Linear Regression — R²: 0.335, RMSE: 3.83

Regression coefficients (standardised):
dribbling    2.537
physic       2.299
defending    1.570
shooting     1.041
pace         0.768
passing      0.252
dtype: float64


In [14]:
fig, ax = plt.subplots(figsize=(9,4))
coef_df.plot(kind='bar', ax=ax, color=['green' if c>0 else 'red' for c in coef_df])
ax.set_title('Regression Coefficients — Predicting Overall Rating')
ax.set_ylabel('Standardised Coefficient')
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Dribbling and passing** are the most positively correlated with overall rating.
2. **Defending** has a lower correlation because it is position-specific.
3. **Linear regression on 6 attributes** achieves R² > 0.7 — strong predictive power.
4. **Position stratification** is critical — attackers and defenders have different attribute profiles.
5. **Age and overall** peak around 27–29 years old for outfield players.

## 20. Limitations
- FIFA ratings are editorial/algorithm-driven, not solely stats-based.
- Six aggregate attributes hide sub-skill details (finishing vs long shots within shooting).
- Goalkeeper ratings use a completely different attribute set.
- Dataset reflects one FIFA edition — ratings change annually.

## 21. Recommendations / Next Steps
1. Filter out goalkeepers before modelling.
2. Use the full sub-skill attribute set (75+ individual skills) for a richer model.
3. Train a position-stratified model (separate models for attackers, midfielders, defenders).
4. Apply Ridge regression to handle multicollinearity among correlated attributes.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Including GKs in outfield analysis | GK attributes are fundamentally different | Filter by position |
| Not scaling before regression | Skews coefficient interpretation | StandardScaler |
| Using R² alone | R² is optimistic; can be high for any fit with many features | Also check RMSE |
| Treating FIFA rating as real-world ground truth | It is a commercial product with editorial bias | Note the caveat |
| Ignoring multicollinearity | Inflates standard errors of correlated predictors | Use Ridge for prediction |

## 23. Mini Challenge / Exercises
1. **Value regression**: Repeat with log(value_eur) as target — which attributes predict player value most?
2. **Age curve**: Plot average overall rating by age — at what age do players peak?
3. **Nationality**: Which country has the highest average overall rating for ≥20 players?
4. **Ridge vs OLS**: Compare Ridge(alpha=10) to OLS — does R² change significantly?
5. **How to extend**: Download FIFA 18–22 datasets and track how player ratings evolve over time.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Dribbling and passing are the top predictors of overall FIFA rating.
TAKEAWAY 2  Linear regression on 6 aggregate stats achieves R²~0.7 — solid baseline.
TAKEAWAY 3  Position stratification is essential — attacking and defensive ratings differ.
TAKEAWAY 4  Player value peaks at a different age than overall rating.
TAKEAWAY 5  FIFA data is a rich playground for regression, clustering, and sports analytics.
```